# 04 - Map Skills to Taxonomy

Notebook này nhận danh sách kỹ năng đã trích xuất từ CV ở bước 03, sau đó ánh xạ từng kỹ năng vào taxonomy đã xây dựng ở DataXomy.

Mục tiêu chính:

1. Đọc danh sách skill đã extract từ `03_cv_skills_extracted.xlsx`.
2. Đọc bảng mapping sạch từ `14_skill_mapping_clean.xlsx`.
3. Đọc thêm alias/tên gọi khác từ file Djinni nếu có.
4. Chuẩn hóa tên skill.
5. Map từng skill về skill chuẩn và nhóm taxonomy.
6. Xuất kết quả ra `04_cv_skills_mapped.xlsx`.

Kết quả của notebook này trả lời câu hỏi:

> Ứng viên có skill gì, skill đó được chuẩn hóa thành tên nào, và thuộc nhóm taxonomy nào?

## 1. Import thư viện

Các thư viện sử dụng:

- `pandas`: đọc/ghi Excel và xử lý bảng dữ liệu.
- `re`: xử lý chuỗi bằng biểu thức chính quy.
- `ast`: đọc list được lưu dưới dạng chuỗi trong Excel.
- `Path`: quản lý đường dẫn file.

In [19]:
import ast
import re
from pathlib import Path

import pandas as pd

## 2. Khai báo đường dẫn

Notebook này cần 3 file đầu vào:

- `03_cv_skills_extracted.xlsx`: danh sách skill đã tìm thấy trong CV.
- `14_skill_mapping_clean.xlsx`: bảng mapping skill đã làm sạch.
- `08_djinni_step4_final.xlsx`: bảng alias/tên gọi mở rộng của skill.

File đầu ra:

- `04_cv_skills_mapped.xlsx`: danh sách skill của từng candidate sau khi map vào taxonomy.

Lưu ý: Có thể chỉnh `BASE_DIR` cho phù hợp với máy đang chạy.

In [20]:
BASE_DIR = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline")

CV_SKILL_INPUT_PATH = BASE_DIR / "data_outputs" / "step03_skill_extract" / "03_cv_skills_extracted.xlsx"
OUTPUT_DIR = BASE_DIR / "data_outputs" / "step04_skill_map"
OUTPUT_PATH = OUTPUT_DIR / "04_cv_skills_mapped.xlsx"

SKILL_MAPPING_PATH = Path(
    "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/14_skill_mapping_clean.xlsx"
)

DJINNI_PATH = Path(
    "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CV skill input:", CV_SKILL_INPUT_PATH, CV_SKILL_INPUT_PATH.exists())
print("Skill mapping:", SKILL_MAPPING_PATH, SKILL_MAPPING_PATH.exists())
print("Djinni alias file:", DJINNI_PATH, DJINNI_PATH.exists())
print("Output:", OUTPUT_PATH)

CV skill input: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step03_skill_extract/03_cv_skills_extracted.xlsx True
Skill mapping: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/ESCO_taxonomy/notebook_clean/14_skill_mapping_clean.xlsx True
Djinni alias file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/DataXomy/Djinni/notebook_clean/08_djinni_step4_final.xlsx True
Output: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step04_skill_map/04_cv_skills_mapped.xlsx


## 3. Đọc dữ liệu đầu vào

Ba dataframe chính:

- `cv_skill_df`: dữ liệu skill đã extract từ CV ở file 03.
- `skill_df`: dữ liệu mapping clean từ file 14.
- `djinni_df`: dữ liệu alias/tên gọi khác của skill.

Trong bước này chỉ đọc dữ liệu và xem nhanh kích thước bảng.

In [21]:
cv_skill_df = pd.read_excel(CV_SKILL_INPUT_PATH, engine="openpyxl")
skill_df = pd.read_excel(SKILL_MAPPING_PATH, engine="openpyxl")
djinni_df = pd.read_excel(DJINNI_PATH, engine="openpyxl")

print("cv_skill_df:", cv_skill_df.shape)
print("skill_df:", skill_df.shape)
print("djinni_df:", djinni_df.shape)

display(cv_skill_df.head(2))
display(skill_df.head(2))
display(djinni_df.head(2))

cv_skill_df: (20, 28)
skill_df: (1171, 10)
djinni_df: (1171, 28)


,candidate_id,cv_text_raw,cv_text_clean,raw_length,clean_length,is_empty_clean,section_summary,section_skills,section_experience,section_education,...,section_certifications_len,section_other_len,matched_skills_skills,matched_skills_experience,matched_skills_projects,matched_skills_other,matched_skills_summary,matched_skills_clean_text,matched_skills_all,n_matched_skills
0,C001,"Frontend developer experienced in ReactJS, Vue...",frontend developer experienced in reactjs vuej...,249,240,False,NaN,NaN,NaN,NaN,...,0,240,[],[],[],"['javascript', 'implement frontend website des...",[],"['javascript', 'implement frontend website des...","['javascript', 'implement frontend website des...",3
1,C002,"Backend engineer experienced in Java, Spring B...",backend engineer experienced in java spring bo...,188,181,False,NaN,NaN,NaN,NaN,...,0,181,[],[],[],"['perform software unit testing', 'mysql']",[],"['perform software unit testing', 'mysql']","['perform software unit testing', 'mysql']",2


,skill_id,skill_name,skill_type,group,relation_count,skill_group,skill_subgroup,mapped_taxonomy_group,mapped_taxonomy_subgroup,notes
0,NaN,Python (computer programming),optional,extended,20,AI / Data Tool,Machine Learning / AI,Data & AI,AI / Machine Learning,NaN
1,NaN,computer vision,optional,core,1,AI / Data Tool,Machine Learning / AI,Data & AI,AI / Machine Learning,NaN


,row_id,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup,ten_goc,ten_sach,ten_ngoai_thi_truong,cac_ten_gan_giong,huong_xu_ly,...,buoc,ten_thi_truong,ten_gan_giong,flag_missing_market_name,flag_missing_alias,market_name_norm,flag_duplicate_market_name,review_priority,review_issue,can_xem_thu_cong
0,2,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,Python (computer programming),python (computer programming),python (computer programming),NaN,giu_nguyen,...,4,python (computer programming),NaN,0,0,python (computer programming),0,NaN,NaN,0
1,3,Data & AI,AI / Machine Learning,AI / Data Tool,Machine Learning / AI,computer vision,computer vision,computer vision,NaN,giu_nguyen,...,4,computer vision,NaN,0,0,computer vision,0,NaN,NaN,0


## 4. Kiểm tra cột bắt buộc của file 03

File `03_cv_skills_extracted.xlsx` bắt buộc phải có:

- `candidate_id`: mã ứng viên.
- `matched_skills_all`: danh sách skill đã tìm thấy trong CV.

Nếu thiếu một trong hai cột này thì không thể map skill cho từng ứng viên.

In [22]:
required_cv_cols = ["candidate_id", "matched_skills_all"]

missing_cv_cols = [
    col for col in required_cv_cols
    if col not in cv_skill_df.columns
]

if missing_cv_cols:
    raise ValueError(f"Thiếu cột trong 03_cv_skills_extracted.xlsx: {missing_cv_cols}")

print("File 03 có đủ cột bắt buộc.")

File 03 có đủ cột bắt buộc.


## 5. Xem nhanh tên cột của file mapping

Tên cột trong các file mapping có thể thay đổi giữa các phiên bản.  
Vì vậy cần in danh sách cột để kiểm tra trước khi tự động chọn cột phù hợp.

In [23]:
print("skill_df columns:")
print(skill_df.columns.tolist())

print("\ndjinni_df columns:")
print(djinni_df.columns.tolist())

skill_df columns:
['skill_id', 'skill_name', 'skill_type', 'group', 'relation_count', 'skill_group', 'skill_subgroup', 'mapped_taxonomy_group', 'mapped_taxonomy_subgroup', 'notes']

djinni_df columns:
['row_id', 'nhom_lon', 'nhom_nho', 'cum_ky_nang', 'skill_subgroup', 'ten_goc', 'ten_sach', 'ten_ngoai_thi_truong', 'cac_ten_gan_giong', 'huong_xu_ly', 'ghi_chu_djinni', 'skill_id', 'skill_type', 'group', 'relation_count', 'notes', 'da_kiem', 'vong', 'buoc', 'ten_thi_truong', 'ten_gan_giong', 'flag_missing_market_name', 'flag_missing_alias', 'market_name_norm', 'flag_duplicate_market_name', 'review_priority', 'review_issue', 'can_xem_thu_cong']


## 6. Hàm chuẩn hóa text

Hai hàm chính:

- `normalize_text`: đưa chuỗi về chữ thường, bỏ khoảng trắng thừa.
- `normalize_skill_phrase`: chuẩn hóa tên skill để so khớp ổn định hơn.

Ví dụ:

- `MySQL` → `mysql`
- ` ReactJS  ` → `reactjs`
- `JavaScript!!!` → `javascript`

In [24]:
def normalize_text(text):
    """Chuẩn hóa text cơ bản: lowercase, strip, gom khoảng trắng."""
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_skill_phrase(text):
    """Chuẩn hóa cụm skill để dùng làm key tra cứu."""
    text = normalize_text(text)
    text = re.sub(r"[^\w\s\+\#\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 7. Tự động nhận diện cột quan trọng trong file 14

File `14_skill_mapping_clean.xlsx` có thể có nhiều cách đặt tên cột.  
Notebook sẽ tự tìm các cột phù hợp theo danh sách ứng viên.

Các nhóm cột cần tìm:

- Cột tên skill chuẩn.
- Cột nhóm lớn.
- Cột nhóm nhỏ.
- Cột cụm kỹ năng.
- Cột subgroup của skill.

Nếu không tìm được cột tên skill chuẩn thì notebook dừng, vì không có dữ liệu để map.

In [25]:
possible_skill_name_cols = [
    "ten_sach", "skill_clean", "preferred_label", "skill_name", "skill", "ten_goc"
]

possible_group_cols = [
    "mapped_taxonomy_group",
    "taxonomy_group",
    "nhom_lon",
    "skill_group",
    "group",
]

possible_subgroup_cols = [
    "mapped_taxonomy_subgroup",
    "taxonomy_subgroup",
    "nhom_nho",
    "skill_subgroup",
]

possible_subgroup_cols = [
    "nhom_nho", "subgroup", "mapped_taxonomy_subgroup"
]

possible_cluster_cols = [
    "cum_ky_nang", "cluster", "skill_group"
]

possible_skill_subgroup_cols = [
    "skill_subgroup"
]

skill_name_col = next((c for c in possible_skill_name_cols if c in skill_df.columns), None)
group_col = next((c for c in possible_group_cols if c in skill_df.columns), None)
subgroup_col = next((c for c in possible_subgroup_cols if c in skill_df.columns), None)
cluster_col = next((c for c in possible_cluster_cols if c in skill_df.columns), None)
skill_subgroup_col = next((c for c in possible_skill_subgroup_cols if c in skill_df.columns), None)

print("skill_name_col =", skill_name_col)
print("group_col =", group_col)
print("subgroup_col =", subgroup_col)
print("cluster_col =", cluster_col)
print("skill_subgroup_col =", skill_subgroup_col)

if skill_name_col is None:
    raise ValueError("Không tìm thấy cột skill chuẩn trong file 14_skill_mapping_clean.xlsx")

skill_name_col = skill_name
group_col = mapped_taxonomy_group
subgroup_col = mapped_taxonomy_subgroup
cluster_col = skill_group
skill_subgroup_col = skill_subgroup


## 8. Tạo `taxonomy_lookup`

`taxonomy_lookup` là dictionary tra cứu từ file 14.

Cấu trúc:

```text
skill đã chuẩn hóa → thông tin taxonomy của skill
```

Ví dụ:

```text
mysql → {
    mapped_skill: MySQL,
    nhom_lon: Data & AI,
    nhom_nho: Database,
    ...
}
```

Mục đích: khi có một skill từ CV, có thể tra nhanh skill đó thuộc nhóm nào dựa vào file 14


In [26]:
taxonomy_lookup = {}

for _, row in skill_df.iterrows():
    raw_skill_name = row.get(skill_name_col, "")
    skill_key = normalize_skill_phrase(raw_skill_name)

    if not skill_key:
        continue

    taxonomy_lookup[skill_key] = {
    "mapped_skill": raw_skill_name,
    "nhom_lon": row.get(group_col, ""),
    "nhom_nho": row.get(subgroup_col, ""),
    "cum_ky_nang": row.get(cluster_col, ""),
    "skill_subgroup": row.get(skill_subgroup_col, ""),
}

print("Số skill trong taxonomy_lookup:", len(taxonomy_lookup))

for item in list(taxonomy_lookup.items())[:5]:
    print(item)

Số skill trong taxonomy_lookup: 1171
('python computer programming', {'mapped_skill': 'Python (computer programming)', 'nhom_lon': 'Data & AI', 'nhom_nho': 'AI / Machine Learning', 'cum_ky_nang': 'AI / Data Tool', 'skill_subgroup': 'Machine Learning / AI'})
('computer vision', {'mapped_skill': 'computer vision', 'nhom_lon': 'Data & AI', 'nhom_nho': 'AI / Machine Learning', 'cum_ky_nang': 'AI / Data Tool', 'skill_subgroup': 'Machine Learning / AI'})
('deep learning', {'mapped_skill': 'deep learning', 'nhom_lon': 'Data & AI', 'nhom_nho': 'AI / Machine Learning', 'cum_ky_nang': 'AI / Data Tool', 'skill_subgroup': 'Machine Learning / AI'})
('machine learning', {'mapped_skill': 'machine learning', 'nhom_lon': 'Data & AI', 'nhom_nho': 'AI / Machine Learning', 'cum_ky_nang': 'AI / Data Tool', 'skill_subgroup': 'Machine Learning / AI'})
('utilise machine learning', {'mapped_skill': 'utilise machine learning', 'nhom_lon': 'Data & AI', 'nhom_nho': 'AI / Machine Learning', 'cum_ky_nang': 'AI / Da

## 9. Xác định cột alias trong file Djinni

File Djinni/mapping có thể chứa nhiều cột tên gọi khác nhau của skill.

Ví dụ:

- `ten_sach`: tên chuẩn.
- `ten_ngoai_thi_truong`: tên skill ngoài thị trường.
- `cac_ten_gan_giong`: các tên gần giống.
- `ten_goc`: tên gốc.

Bước này tìm cột nào dùng làm skill chuẩn và cột nào dùng làm alias.

In [27]:
possible_djinni_canonical_cols = [
    "ten_sach", "skill_clean", "skill_name", "ten_goc"
]

possible_alias_cols = [
    "ten_ngoai_thi_truong", "cac_ten_gan_giong", "ten_goc"
]

djinni_canonical_col = next(
    (c for c in possible_djinni_canonical_cols if c in djinni_df.columns),
    None
)

djinni_alias_cols = [
    c for c in possible_alias_cols
    if c in djinni_df.columns
]

print("djinni_canonical_col =", djinni_canonical_col)
print("djinni_alias_cols =", djinni_alias_cols)

djinni_canonical_col = ten_sach
djinni_alias_cols = ['ten_ngoai_thi_truong', 'cac_ten_gan_giong', 'ten_goc']


## 10. Hàm parse alias

Alias trong Excel có thể được lưu theo nhiều kiểu:

- Dạng list string: `["js", "javascript"]`
- Dạng chuỗi phân tách bằng dấu `;`, `,`, hoặc `|`
- Dạng một giá trị đơn

Hàm `parse_alias_value` đưa các kiểu trên về list Python.

In [28]:
def parse_alias_value(value):
    """Chuyển một ô alias trong Excel thành list alias."""
    if pd.isna(value):
        return []

    value = str(value).strip()
    if not value:
        return []

    # Trường hợp alias được lưu dạng list string: ["js", "javascript"]
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    # Trường hợp alias được phân tách bằng ; , |
    parts = re.split(r"[;,|]", value)
    return [p.strip() for p in parts if p.strip()]

## 11. Tạo `alias_to_canonical`

`alias_to_canonical` là dictionary tra cứu alias.

Nó tạo từ điển:

tên gọi khác → tên chuẩn

Ví dụ:

- reactjs → react
- postgres → postgresql
- js → javascript

Tại sao cần?
Vì CV có thể ghi postgres, nhưng file 14 lại lưu chuẩn là postgresql. Nếu không có từ điển này thì không tra được.

In [29]:
alias_to_canonical = {}

if djinni_canonical_col is not None:
    for _, row in djinni_df.iterrows():
        canonical_raw = row.get(djinni_canonical_col, "")
        canonical_norm = normalize_skill_phrase(canonical_raw)

        if not canonical_norm:
            continue

        # Tên chuẩn cũng được xem là một alias của chính nó
        alias_to_canonical[canonical_norm] = canonical_norm

        for alias_col in djinni_alias_cols:
            raw_alias_value = row.get(alias_col, "")
            alias_list = parse_alias_value(raw_alias_value)

            if not alias_list and pd.notna(raw_alias_value):
                alias_list = [str(raw_alias_value).strip()]

            for alias in alias_list:
                alias_norm = normalize_skill_phrase(alias)
                if alias_norm:
                    alias_to_canonical[alias_norm] = canonical_norm

print("Số alias trong Djinni:", len(alias_to_canonical))
list(alias_to_canonical.items())[:10]

Số alias trong Djinni: 1368


[('python computer programming', 'python computer programming'),
 ('computer vision', 'computer vision'),
 ('deep learning', 'deep learning'),
 ('machine learning', 'machine learning'),
 ('ml', 'machine learning'),
 ('ai', 'machine learning'),
 ('model training', 'machine learning'),
 ('utilise machine learning', 'utilise machine learning'),
 ('frostbite digital game creation systems',
  'frostbite digital game creation systems'),
 ('ict accessibility standards', 'ict accessibility standards')]

## 12. Hàm parse danh sách skill từ file 03

Nó lấy cột matched_skills_all từ file 03 và biến về list thật.

Ví dụ trong Excel có:

["mysql", "postgres", "reactjs"]

Nhưng Excel đang lưu nó như chuỗi text, không phải list thật.

Cell 12 biến nó thành:

["mysql", "postgres", "reactjs"]

Tại sao cần?
Vì muốn map từng skill thì phải có list để loop.

In [30]:
def parse_skill_list(value):
    """Chuyển matched_skills_all thành list skill."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()
    if not value:
        return []

    # Trường hợp list bị lưu thành string
    if value.startswith("[") and value.endswith("]"):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass

    # Trường hợp skill được phân tách bằng ; , |
    parts = re.split(r"[;,|]", value)
    return [p.strip() for p in parts if p.strip()]

## 13. Hàm map một skill vào taxonomy

Đây là hàm map 1 skill.

Ví dụ nhận vào:

postgres

Nó làm:

1. Tra taxonomy_lookup xem có "postgres" không
2. Không có thì tra alias_to_canonical
3. Thấy postgres → postgresql
4. Lấy postgresql tra taxonomy_lookup
5. Trả ra PostgreSQL thuộc nhóm Database

Nếu skill là:

mysql

và mysql có sẵn trong taxonomy_lookup, thì nó trả luôn:

mysql → MySQL → Database

In [31]:
def empty_mapping_result(raw_skill, source="empty"):
    """Tạo kết quả mặc định khi skill rỗng hoặc không map được."""
    return {
        "raw_skill": raw_skill,
        "mapped_skill": "",
        "mapping_source": source,
        "mapping_confidence": 0.0,
        "nhom_lon": "",
        "nhom_nho": "",
        "cum_ky_nang": "",
        "skill_subgroup": "",
    }


def build_mapping_result(raw_skill, meta, source, confidence):
    """Tạo kết quả map thành công."""
    return {
        "raw_skill": raw_skill,
        "mapped_skill": meta["mapped_skill"],
        "mapping_source": source,
        "mapping_confidence": confidence,
        "nhom_lon": meta["nhom_lon"],
        "nhom_nho": meta["nhom_nho"],
        "cum_ky_nang": meta["cum_ky_nang"],
        "skill_subgroup": meta["skill_subgroup"],
    }


def map_skill_to_taxonomy(raw_skill):
    """Map một skill thô từ CV vào taxonomy."""
    raw_skill_norm = normalize_skill_phrase(raw_skill)

    if not raw_skill_norm:
        return empty_mapping_result(raw_skill, source="empty")

    # 1. Match trực tiếp với skill clean trong file 14
    if raw_skill_norm in taxonomy_lookup:
        meta = taxonomy_lookup[raw_skill_norm]
        return build_mapping_result(
            raw_skill=raw_skill,
            meta=meta,
            source="exact_clean",
            confidence=1.0,
        )

    # 2. Match qua alias từ Djinni
    if raw_skill_norm in alias_to_canonical:
        canonical_norm = alias_to_canonical[raw_skill_norm]

        if canonical_norm in taxonomy_lookup:
            meta = taxonomy_lookup[canonical_norm]
            return build_mapping_result(
                raw_skill=raw_skill,
                meta=meta,
                source="djinni_alias",
                confidence=0.95,
            )

    # 3. Không map được
    return {
        "raw_skill": raw_skill,
        "mapped_skill": raw_skill,
        "mapping_source": "unmapped",
        "mapping_confidence": 0.0,
        "nhom_lon": "",
        "nhom_nho": "",
        "cum_ky_nang": "",
        "skill_subgroup": "",
    }

## 14. Map toàn bộ skill của từng candidate

Bước này chạy qua từng dòng trong `cv_skill_df`.

Với mỗi candidate:

1. Lấy `matched_skills_all`.
2. Parse thành list skill.
3. Map từng skill bằng hàm `map_skill_to_taxonomy`.
4. Lưu kết quả vào `mapped_df`.

Sau bước này, mỗi dòng trong `mapped_df` tương ứng với một skill của một candidate.

In [32]:
rows = []

for _, row in cv_skill_df.iterrows():
    candidate_id = row["candidate_id"]
    skill_list = parse_skill_list(row["matched_skills_all"])

    if not skill_list:
        empty_row = empty_mapping_result("", source="no_skill_detected")
        empty_row["candidate_id"] = candidate_id
        rows.append(empty_row)
        continue

    # Loại trùng trong cùng candidate nhưng vẫn giữ thứ tự xuất hiện
    seen = set()
    unique_skills = []
    for skill in skill_list:
        skill_key = normalize_skill_phrase(skill)
        if skill_key and skill_key not in seen:
            seen.add(skill_key)
            unique_skills.append(skill)

    for skill in unique_skills:
        mapped = map_skill_to_taxonomy(skill)
        mapped["candidate_id"] = candidate_id
        rows.append(mapped)

mapped_df = pd.DataFrame(rows)

# Đưa candidate_id lên đầu bảng cho dễ đọc
ordered_cols = ["candidate_id"] + [col for col in mapped_df.columns if col != "candidate_id"]
mapped_df = mapped_df[ordered_cols]

print("mapped_df shape:", mapped_df.shape)
display(mapped_df.head(20))

mapped_df shape: (38, 9)


,candidate_id,raw_skill,mapped_skill,mapping_source,mapping_confidence,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup
0,C001,javascript,JavaScript,exact_clean,1.0,Software Development,Frontend,Programming Languages,Web Development
1,C001,implement frontend website design,implement frontend website design,unmapped,0.0,,,,
2,C001,css,CSS,exact_clean,1.0,Software Development,Frontend,Programming Languages,Web Development
3,C002,perform software unit testing,perform software unit testing,exact_clean,1.0,Software Development,Code Quality & Review,Software Development,Software Testing
4,C002,mysql,MySQL,exact_clean,1.0,Software Development,Backend,Databases,Database Management
5,C003,perform data analysis,perform data analysis,exact_clean,1.0,Data & Databases,Data Analysis,Data & Analytics,Data Analysis
6,C003,sql,SQL,exact_clean,1.0,Data & Databases,Query Languages,Data & Databases,Query Languages & Analytics
7,C004,web services,web services,exact_clean,1.0,IT Infrastructure & Operations,System Integration,Systems Integration & Middleware,Web Services / APIs
8,C004,devops,DevOps,exact_clean,1.0,IT Infrastructure & Operations,DevOps,DevOps & Infrastructure,DevOps
9,C005,,,no_skill_detected,0.0,,,,


## 15. Kiểm tra nguồn mapping

Cột `mapping_source` cho biết mỗi skill được map bằng cách nào:

- `exact_clean`: match trực tiếp với file 14.
- `djinni_alias`: match qua alias rồi quay về file 14.
- `unmapped`: chưa map được.
- `no_skill_detected`: candidate không có skill nào được extract ở bước 03.

In [33]:
print(mapped_df["mapping_source"].value_counts(dropna=False))

mapping_source
exact_clean          31
unmapped              4
no_skill_detected     3
Name: count, dtype: int64


## 16. Xem các skill chưa map được

Bước này giúp phát hiện skill nào cần bổ sung vào mapping trong tương lai.

Nếu có nhiều skill `unmapped`, có thể cần cập nhật:

- `12_skill_master.xlsx`
- `14_skill_mapping_clean.xlsx`
- file alias Djinni/mapping

In [34]:
unmapped_df = mapped_df[mapped_df["mapping_source"] == "unmapped"].copy()

print("Số skill chưa map được:", len(unmapped_df))
display(unmapped_df.head(20))

Số skill chưa map được: 4


,candidate_id,raw_skill,mapped_skill,mapping_source,mapping_confidence,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup
1,C001,implement frontend website design,implement frontend website design,unmapped,0.0,,,,
19,C012,database,database,unmapped,0.0,,,,
22,C013,implement frontend website design,implement frontend website design,unmapped,0.0,,,,
35,C020,database,database,unmapped,0.0,,,,


## 17. Lưu output

Xuất kết quả cuối cùng ra:

```text
04_cv_skills_mapped.xlsx
```

File này là đầu vào cho bước 05 để xây dựng candidate profile.

In [35]:
mapped_df.to_excel(OUTPUT_PATH, index=False)
print("Đã lưu file:", OUTPUT_PATH)

Đã lưu file: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ProcessPipeline/data_outputs/step04_skill_map/04_cv_skills_mapped.xlsx


## 18. Đọc lại output để kiểm tra

Sau khi lưu, đọc lại file output để đảm bảo file được ghi đúng.

In [18]:
result = pd.read_excel(OUTPUT_PATH, engine="openpyxl")

print("Output shape:", result.shape)
display(result.head(20))

Output shape: (36, 9)


,candidate_id,raw_skill,mapped_skill,mapping_source,mapping_confidence,nhom_lon,nhom_nho,cum_ky_nang,skill_subgroup
0,C001,perform software unit testing,perform software unit testing,exact_clean,1,Software Development,Code Quality & Review,Software Development,Software Testing
1,C001,mysql,MySQL,exact_clean,1,Software Development,Backend,Databases,Database Management
2,C002,javascript,JavaScript,exact_clean,1,Software Development,Frontend,Programming Languages,Web Development
3,C002,typescript,TypeScript,exact_clean,1,Software Development,Frontend,Programming Languages,Web Development
4,C002,implement frontend website design,implement frontend website design,unmapped,0,NaN,NaN,NaN,NaN
5,C003,sql,SQL,exact_clean,1,Data & Databases,Query Languages,Data & Databases,Query Languages & Analytics
6,C003,perform data analysis,perform data analysis,exact_clean,1,Data & Databases,Data Analysis,Data & Analytics,Data Analysis
7,C004,devops,DevOps,exact_clean,1,IT Infrastructure & Operations,DevOps,DevOps & Infrastructure,DevOps
8,C005,NaN,NaN,no_skill_detected,0,NaN,NaN,NaN,NaN
9,C006,NaN,NaN,no_skill_detected,0,NaN,NaN,NaN,NaN


## Tóm tắt luồng xử lý

```text
03_cv_skills_extracted.xlsx
        ↓
lấy matched_skills_all
        ↓
14_skill_mapping_clean.xlsx → tạo taxonomy_lookup
08_djinni_step4_final.xlsx → tạo alias_to_canonical
        ↓
map từng raw_skill
        ↓
04_cv_skills_mapped.xlsx
```

Ý nghĩa ngắn gọn:

Nếu như taxonomy_lok không có thì quay sang alias to canonical tại vì có thể CV sẽ để position mong muốn là tên đúng hoặc là tên ngoài thị trường. 

Kết quả là mỗi skill trong CV được chuẩn hóa và gán vào nhóm taxonomy tương ứng. File 04 chưa kết luận toàn bộ CV thuộc nhóm nào, mà chỉ map từng 

skill; bước 05 mới gom các skill đó để build candidate profile.